### Glorot Initialization and He Initialization

In [1]:
import torch
import torch.nn as nn

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5   # Kaiming init (or 3 ** 0.5 for LeCun init)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [2]:
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [3]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

model = nn.Sequential(nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1), nn.ReLU())
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=50, out_features=40, bias=True)
  (1): ReLU()
  (2): Linear(in_features=40, out_features=1, bias=True)
  (3): ReLU()
)

### Better Activation Functions

In [4]:
alpha = 0.2
model = nn.Sequential(nn.Linear(50, 40), nn.LeakyReLU(negative_slope=alpha))
nn.init.kaiming_uniform_(model[0].weight, alpha, nonlinearity="leaky_relu")

Parameter containing:
tensor([[ 0.3194,  0.0445,  0.1882,  ..., -0.2450, -0.2021,  0.2694],
        [ 0.0748,  0.0655, -0.2901,  ...,  0.2733, -0.1493, -0.2521],
        [-0.0788,  0.0959, -0.2000,  ..., -0.3221,  0.1271, -0.2279],
        ...,
        [-0.3072,  0.3143,  0.3282,  ...,  0.0773, -0.2581,  0.0309],
        [ 0.2283, -0.1092,  0.0771,  ...,  0.0238,  0.0044,  0.1719],
        [-0.2479,  0.0203,  0.3068,  ...,  0.1134, -0.3269, -0.1206]],
       requires_grad=True)

### Batch Normalization

In [5]:
# Fashion-MNIST classifier using Batch-Norm

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

### Layer Normalization

In [6]:
inputs = torch.randn(32, 3, 100, 200)    # a batch of random RGB images
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)    # normalizes over the last two dimensions
result.shape

torch.Size([32, 3, 100, 200])

In [7]:
layer_norm.weight.shape

torch.Size([100, 200])

In [8]:
layer_norm.bias.shape

torch.Size([100, 200])

In [9]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)   # normalizes over the last three dimensions
result.shape

torch.Size([32, 3, 100, 200])

In [10]:
layer_norm.weight.shape

torch.Size([3, 100, 200])

In [11]:
layer_norm.bias.shape

torch.Size([3, 100, 200])

### Transfer Learning with PyTorch

In [12]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml

fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False)
X = torch.FloatTensor(fashion_mnist.data.reshape(-1, 1, 28, 28) / 255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y == 0) | (y == 2)  # Pullover or T-shirt/top
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))  # [1,3,4,5,6,7,8,9] => [0,..,7]
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1, 1)

train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:], y_B[5000:])


In [13]:
batch_size = 256
train_loader_A = DataLoader(train_set_A, batch_size=batch_size, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=batch_size)
test_loader_A = DataLoader(test_set_A, batch_size=batch_size)
train_loader_B = DataLoader(train_set_B, batch_size=batch_size, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=batch_size)
test_loader_B = DataLoader(test_set_B, batch_size=batch_size)

In [14]:
def evaluate_tm(model, data_loader, metric, device=None):
    if device:
        model = model.to(device)
        metric = metric.to(device)
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

def train(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, device=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric, device=device).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [15]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8)
)

In [16]:
model_A.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=8, bias=True)
)

In [17]:
import torchmetrics

device="mps"
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train(model_A, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs, device=device)

Epoch 1/20, train loss: 1.2990, train metric: 0.5904, valid metric: 0.7255
Epoch 2/20, train loss: 0.7052, train metric: 0.7583, valid metric: 0.7840
Epoch 3/20, train loss: 0.5654, train metric: 0.8053, valid metric: 0.8195
Epoch 4/20, train loss: 0.4950, train metric: 0.8352, valid metric: 0.8410
Epoch 5/20, train loss: 0.4474, train metric: 0.8533, valid metric: 0.8440
Epoch 6/20, train loss: 0.4138, train metric: 0.8640, valid metric: 0.8630
Epoch 7/20, train loss: 0.3885, train metric: 0.8706, valid metric: 0.8670
Epoch 8/20, train loss: 0.3693, train metric: 0.8753, valid metric: 0.8710
Epoch 9/20, train loss: 0.3538, train metric: 0.8808, valid metric: 0.8745
Epoch 10/20, train loss: 0.3409, train metric: 0.8848, valid metric: 0.8750
Epoch 11/20, train loss: 0.3303, train metric: 0.8881, valid metric: 0.8855
Epoch 12/20, train loss: 0.3216, train metric: 0.8903, valid metric: 0.8795
Epoch 13/20, train loss: 0.3141, train metric: 0.8930, valid metric: 0.8595
Epoch 14/20, train lo

In [18]:
torch.manual_seed(42)

model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1)
)
model_B.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=1, bias=True)
)

In [19]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary")
history_B = train(model_B, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs, device=device)

Epoch 1/20, train loss: 0.7110, train metric: 0.5000, valid metric: 0.5369
Epoch 2/20, train loss: 0.6745, train metric: 0.5000, valid metric: 0.5815
Epoch 3/20, train loss: 0.6400, train metric: 0.7000, valid metric: 0.6285
Epoch 4/20, train loss: 0.6101, train metric: 0.8500, valid metric: 0.6657
Epoch 5/20, train loss: 0.5823, train metric: 0.9000, valid metric: 0.6920
Epoch 6/20, train loss: 0.5559, train metric: 0.9000, valid metric: 0.7169
Epoch 7/20, train loss: 0.5311, train metric: 0.9500, valid metric: 0.7341
Epoch 8/20, train loss: 0.5076, train metric: 1.0000, valid metric: 0.7462
Epoch 9/20, train loss: 0.4859, train metric: 1.0000, valid metric: 0.7576
Epoch 10/20, train loss: 0.4656, train metric: 1.0000, valid metric: 0.7687
Epoch 11/20, train loss: 0.4466, train metric: 1.0000, valid metric: 0.7787
Epoch 12/20, train loss: 0.4290, train metric: 1.0000, valid metric: 0.7839
Epoch 13/20, train loss: 0.4119, train metric: 1.0000, valid metric: 0.7908
Epoch 14/20, train lo

In [20]:
evaluate_tm(model_B, test_loader_B, accuracy, device=device)

tensor(0.8311, device='mps:0')

In [21]:
import copy

torch.manual_seed(42)
reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(
    *reused_layers,
    nn.Linear(100, 1)
)

In [22]:
for layer in model_B_on_A[:-1]:
    for param in layer.parameters():
        param.requires_grad = False

In [23]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary")
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs, device=device)

Epoch 1/10, train loss: 0.6875, train metric: 0.4000, valid metric: 0.7950
Epoch 2/10, train loss: 0.5465, train metric: 0.9000, valid metric: 0.8594
Epoch 3/10, train loss: 0.5320, train metric: 0.8500, valid metric: 0.8685
Epoch 4/10, train loss: 0.5243, train metric: 0.9000, valid metric: 0.8733
Epoch 5/10, train loss: 0.5169, train metric: 0.9000, valid metric: 0.8783
Epoch 6/10, train loss: 0.5098, train metric: 0.9000, valid metric: 0.8815
Epoch 7/10, train loss: 0.5029, train metric: 0.9000, valid metric: 0.8851
Epoch 8/10, train loss: 0.4962, train metric: 0.9000, valid metric: 0.8878
Epoch 9/10, train loss: 0.4897, train metric: 0.9000, valid metric: 0.8930
Epoch 10/10, train loss: 0.4834, train metric: 0.9000, valid metric: 0.8956


In [24]:
# Unfreeze lower layers and fine-tune for a few epochs
for layer in model_B_on_A[2:]:
    for param in layer.parameters():
        param.requires_grad = True

In [25]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.004)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary")
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs, device=device)

Epoch 1/20, train loss: 0.4773, train metric: 0.9000, valid metric: 0.8976
Epoch 2/20, train loss: 0.4707, train metric: 0.9000, valid metric: 0.8990
Epoch 3/20, train loss: 0.4644, train metric: 0.9000, valid metric: 0.9018
Epoch 4/20, train loss: 0.4582, train metric: 0.9000, valid metric: 0.9040
Epoch 5/20, train loss: 0.4522, train metric: 0.9000, valid metric: 0.9056
Epoch 6/20, train loss: 0.4463, train metric: 0.9000, valid metric: 0.9088
Epoch 7/20, train loss: 0.4406, train metric: 0.9000, valid metric: 0.9110
Epoch 8/20, train loss: 0.4350, train metric: 0.9500, valid metric: 0.9114
Epoch 9/20, train loss: 0.4296, train metric: 0.9500, valid metric: 0.9133
Epoch 10/20, train loss: 0.4242, train metric: 0.9500, valid metric: 0.9141
Epoch 11/20, train loss: 0.4189, train metric: 0.9500, valid metric: 0.9143
Epoch 12/20, train loss: 0.4138, train metric: 0.9500, valid metric: 0.9155
Epoch 13/20, train loss: 0.4088, train metric: 0.9500, valid metric: 0.9157
Epoch 14/20, train lo

In [26]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy, device=device)

tensor(0.9249, device='mps:0')

### Faster Optimizers

In [27]:
# Momentum
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.05)

In [28]:
# Nesterov
optimizer = torch.optim.SGD(model.parameters(), nesterov=True, momentum=0.9, lr=0.05)

In [29]:
# RMSProp
optimizer = torch.optim.RMSprop(model.parameters(), alpha=0.9)

In [30]:
# Adam
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)

### Learning rate scheduling

In [31]:
# ExponentialLR
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [32]:
def train_with_scheduler(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, device=None, scheduler=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric, device=device).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
        if scheduler:
            scheduler.step()
    return history

In [33]:
n_epochs = 20
model_A.apply(use_he_init)
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train_with_scheduler(model_A, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs, device=device, scheduler=scheduler)

Epoch 1/20, train loss: 0.6209, train metric: 0.7853, valid metric: 0.7795
Epoch 2/20, train loss: 0.3586, train metric: 0.8720, valid metric: 0.8795
Epoch 3/20, train loss: 0.3046, train metric: 0.8923, valid metric: 0.8900
Epoch 4/20, train loss: 0.2836, train metric: 0.8991, valid metric: 0.9015
Epoch 5/20, train loss: 0.2650, train metric: 0.9065, valid metric: 0.9040
Epoch 6/20, train loss: 0.2499, train metric: 0.9128, valid metric: 0.9030
Epoch 7/20, train loss: 0.2383, train metric: 0.9154, valid metric: 0.9005
Epoch 8/20, train loss: 0.2311, train metric: 0.9179, valid metric: 0.9055
Epoch 9/20, train loss: 0.2236, train metric: 0.9207, valid metric: 0.9090
Epoch 10/20, train loss: 0.2189, train metric: 0.9226, valid metric: 0.9075
Epoch 11/20, train loss: 0.2133, train metric: 0.9259, valid metric: 0.9095
Epoch 12/20, train loss: 0.2098, train metric: 0.9259, valid metric: 0.9040
Epoch 13/20, train loss: 0.2051, train metric: 0.9274, valid metric: 0.9160
Epoch 14/20, train lo

In [34]:
# Cosine Annealing
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=0.001)

In [35]:
# ReduceLrOnPlateau: Monitor validation accuracy (hence mode="max")
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.1)

In [36]:
def train_with_scheduler(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, device=None, scheduler=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric, device=device).item()
        history["valid_metrics"].append(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_metric)
        else:
            scheduler.step()
        print(f"Learning rate: {scheduler.get_last_lr()[0]:.5f}")
    return history

In [37]:
n_epochs = 20
model_A.apply(use_he_init)
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.1)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train_with_scheduler(model_A, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs, device=device, scheduler=scheduler)

Epoch 1/20, train loss: 0.6188, train metric: 0.7829, valid metric: 0.8365
Learning rate: 0.05000
Epoch 2/20, train loss: 0.3623, train metric: 0.8716, valid metric: 0.8190
Learning rate: 0.05000
Epoch 3/20, train loss: 0.3092, train metric: 0.8898, valid metric: 0.8905
Learning rate: 0.05000
Epoch 4/20, train loss: 0.2807, train metric: 0.9018, valid metric: 0.9020
Learning rate: 0.05000
Epoch 5/20, train loss: 0.2575, train metric: 0.9089, valid metric: 0.9115
Learning rate: 0.05000
Epoch 6/20, train loss: 0.2456, train metric: 0.9129, valid metric: 0.9055
Learning rate: 0.05000
Epoch 7/20, train loss: 0.2335, train metric: 0.9172, valid metric: 0.8915
Learning rate: 0.05000
Epoch 8/20, train loss: 0.2235, train metric: 0.9199, valid metric: 0.9060
Learning rate: 0.00500
Epoch 9/20, train loss: 0.2003, train metric: 0.9299, valid metric: 0.9150
Learning rate: 0.00500
Epoch 10/20, train loss: 0.1981, train metric: 0.9307, valid metric: 0.9185
Learning rate: 0.00500
Epoch 11/20, train 

In [38]:
# Warm-up
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=3
)

In [39]:
# Custom
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda epoch: (min(epoch, 3) / 3) * (1.0 - 0.1) + 0.1)

In [40]:
def train_with_warmup(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, device=None, scheduler=None, warmup_scheduler=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        if warmup_scheduler:
            warmup_scheduler.step()
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric, device=device).item()
        history["valid_metrics"].append(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            if epoch >=3:  # deactivate scheduler during warmup
                scheduler.step(val_metric)
        else:
            if epoch >= 3:
                scheduler.step()
        print(f"Learning rate: {scheduler.get_last_lr()[0]:.5f}")
    return history

In [41]:
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.1)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=3
)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train_with_warmup(model_A, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs, device=device, scheduler=scheduler, warmup_scheduler=warmup_scheduler)

Epoch 1/20, train loss: 0.1930, train metric: 0.9317, valid metric: 0.9220
Learning rate: 0.05000
Epoch 2/20, train loss: 0.1951, train metric: 0.9308, valid metric: 0.9185
Learning rate: 0.05000
Epoch 3/20, train loss: 0.2084, train metric: 0.9246, valid metric: 0.9160
Learning rate: 0.05000
Epoch 4/20, train loss: 0.1979, train metric: 0.9297, valid metric: 0.9135
Learning rate: 0.05000
Epoch 5/20, train loss: 0.1908, train metric: 0.9319, valid metric: 0.9185
Learning rate: 0.05000
Epoch 6/20, train loss: 0.1868, train metric: 0.9327, valid metric: 0.9080
Learning rate: 0.05000
Epoch 7/20, train loss: 0.1827, train metric: 0.9337, valid metric: 0.9140
Learning rate: 0.05000
Epoch 8/20, train loss: 0.1778, train metric: 0.9368, valid metric: 0.9115
Learning rate: 0.00500
Epoch 9/20, train loss: 0.1591, train metric: 0.9441, valid metric: 0.9265
Learning rate: 0.00500
Epoch 10/20, train loss: 0.1573, train metric: 0.9448, valid metric: 0.9235
Learning rate: 0.00500
Epoch 11/20, train 

In [42]:
# Cosine annealing with warm restarts
cosine_repeat_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=2, T_mult=2, eta_min=0.001
)

### Regularization

In [43]:
# l2 regularization with SGD optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, weight_decay=1e-4)

In [44]:
# Manual implementation so as not to touch bias and BN parameters
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
params_to_regularize = [
    param for name, param in model.named_parameters()
    if not "bias" in name and not "bn" in name
]


def train_with_l2_regularization(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, params_to_regularize, device=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            main_loss = criterion(y_pred, y_batch)
            l2_loss = sum(param.pow(2.0).sum() for param in params_to_regularize)
            loss = main_loss + 1e-4 * l2_loss
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric, device=device).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [45]:
# With parameter groups passed to the optimizer
param_bias_and_bn = [
    param for name, param in model.named_parameters()
    if "bias" in name or "bn" in name
]
optimizer = torch.optim.SGD([
    {"params": params_to_regularize, "weight_decay": 1e-4},
    {"params": param_bias_and_bn},
], lr=0.05)


In [46]:
# l1 regularization
def train_with_l1_regularization(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, params_to_regularize, device=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            main_loss = criterion(y_pred, y_batch)
            l1_loss = sum(param.abs().sum() for param in params_to_regularize)
            loss = main_loss + 1e-4 * l1_loss
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric, device=device).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

### Dropout

In [47]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(p=0.2), nn.Linear(1 * 28 * 28, 100), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(100, 100), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(100, 100), nn.ReLU(),
    nn.Dropout(p=0.2), nn.Linear(100, 8)
)

In [48]:
model.apply(use_he_init)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train(model, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs, device=device)

Epoch 1/20, train loss: 0.8965, train metric: 0.6587, valid metric: 0.8430
Epoch 2/20, train loss: 0.4883, train metric: 0.8258, valid metric: 0.8605
Epoch 3/20, train loss: 0.4206, train metric: 0.8522, valid metric: 0.8950
Epoch 4/20, train loss: 0.3845, train metric: 0.8644, valid metric: 0.8945
Epoch 5/20, train loss: 0.3613, train metric: 0.8729, valid metric: 0.8960
Epoch 6/20, train loss: 0.3447, train metric: 0.8787, valid metric: 0.9035
Epoch 7/20, train loss: 0.3302, train metric: 0.8843, valid metric: 0.9055
Epoch 8/20, train loss: 0.3208, train metric: 0.8856, valid metric: 0.9020
Epoch 9/20, train loss: 0.3151, train metric: 0.8898, valid metric: 0.9060
Epoch 10/20, train loss: 0.3024, train metric: 0.8940, valid metric: 0.9040
Epoch 11/20, train loss: 0.2988, train metric: 0.8945, valid metric: 0.9085
Epoch 12/20, train loss: 0.2895, train metric: 0.8982, valid metric: 0.9120
Epoch 13/20, train loss: 0.2917, train metric: 0.8975, valid metric: 0.9090
Epoch 14/20, train lo

In [49]:
# MC Dropout
model.eval()
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

X_new = torch.FloatTensor(fashion_mnist.data[:3].reshape(3, 1, 28, 28) / 255)
X_new = X_new.to(device)

torch.manual_seed(42)
with torch.no_grad():
    X_new_repeated = X_new.repeat_interleave(100, dim=0)
    y_logits_all = model(X_new_repeated).reshape(3, 100, 8)
    y_probas_all = torch.nn.functional.softmax(y_logits_all, dim=-1)
    y_probas = y_probas_all.mean(dim=1)


In [50]:
y_probas.round(decimals=2)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0300, 0.0000, 0.9700],
        [0.0000, 0.0500, 0.0000, 0.0000, 0.9500, 0.0000, 0.0000, 0.0000],
        [0.0700, 0.4700, 0.1300, 0.0000, 0.3200, 0.0000, 0.0100, 0.0000]],
       device='mps:0')

In [51]:
y_std = y_probas_all.std(dim=1)
y_std.round(decimals=2)

tensor([[0.0000, 0.0000, 0.0000, 0.0200, 0.0000, 0.0500, 0.0000, 0.0500],
        [0.0000, 0.0800, 0.0100, 0.0000, 0.0900, 0.0000, 0.0100, 0.0000],
        [0.1100, 0.1800, 0.1100, 0.0000, 0.1500, 0.0000, 0.0100, 0.0000]],
       device='mps:0')

In [52]:
# custom module
import torch.nn.functional as F

class McDropout(nn.Dropout):
    def forward(self, input):
        return F.dropout(input, self.p, training=True)

### Max-Norm regularization

In [54]:
# call apply_max_norm(model) in the training loop, 
# right after calling the optimizer’s step() method.
def apply_max_norm(model, max_norm=2, epsilon=1e-8, dim=1):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'bias' not in name:
                actual_norm = param.norm(p=2, dim=dim, keepdim=True)
                target_norm = torch.clamp(actual_norm, 0, max_norm)
                param *= target_norm / (epsilon + actual_norm)